# Data Preprocessing

## Import Libraries

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from imblearn.under_sampling import RandomUnderSampler
import scipy.stats

#FLAGS
undersampling_flag = False

## Read the Data

In [3]:
client_train = pd.read_csv('data/client_train.csv', low_memory=False)
invoice_train = pd.read_csv('data/invoice_train.csv', low_memory=False)

client_test = pd.read_csv(f'data/client_test.csv', low_memory=False)
invoice_test = pd.read_csv(f'data/invoice_test.csv', low_memory=False)
#sample_submission = pd.read_csv(f'results/SampleSubmission.csv', low_memory=False)

In [4]:
client_train.head()

,disrict,client_id,client_catg,region,creation_date,target
0,60,train_Client_0,11,101,31/12/1994,0.0
1,69,train_Client_1,11,107,29/05/2002,0.0
2,62,train_Client_10,11,301,13/03/1986,0.0
3,69,train_Client_100,11,105,11/07/1996,0.0
4,62,train_Client_1000,11,303,14/10/2014,0.0


## Data Cleaning

In [5]:
train = invoice_train.copy()
# counter_statue == 269375 -> can be droped, since there is just one observation and also the train_Client_53725 belong to only that observation 
train = train.query("counter_statue != 269375")
# counter_statue == 420 and reading_remarque == 5 and counter_code == 1 -> they are just one time in the same observation, so we can drop them as well
train = train.query("counter_statue != 420")
# remove ounter_statue with A
train = train.query("counter_statue != 'A'")

# replace string of int to int
train['counter_statue'] = train['counter_statue'].replace('0', 0)
train['counter_statue'] = train['counter_statue'].replace('1', 1)
train['counter_statue'] = train['counter_statue'].replace('4', 4)
train['counter_statue'] = train['counter_statue'].replace('5', 5)
# counter_statue == 769 and reading_remarque == 207 ???
train = train.query("counter_statue != 769")
# counter_statue == 618 and reading_remarque == 413 ???
train = train.query("counter_statue != 618")
# counter_statue == 46 and reading_remarque == 203 ???
train = train.query("counter_statue != 46")
#months_number > 88
train = train.query("months_number <= 88") # auch aus testdaten
invoice_train = train.copy()

invoice_test = invoice_test.query("months_number <= 88")

## Train Test Split

In [6]:
# split client train, stratify by target variable, and set random state for reproducibility
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(client_train.drop(columns=['target']), client_train['target'], test_size=0.2, stratify=client_train['target'], random_state=42)

In [7]:
print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)

(108394, 5) (108394,)
(27099, 5) (27099,)


In [8]:
client_train = pd.merge(X_train, y_train, left_index=True, right_index=True)
client_val = pd.merge(X_val, y_val, left_index=True, right_index=True)

### undersampling

In [9]:
# undersample the majority class in the training data if requested
if undersampling_flag:
    rus = RandomUnderSampler(
        sampling_strategy=0.2,
        random_state=42
    )
    X_train_res, y_train_res = rus.fit_resample(X_train, y_train)
else:
    # keep the original training split when undersampling is disabled
    X_train_res = X_train.copy()
    y_train_res = y_train.copy()

In [10]:
clients_in_resampled_train = X_train_res['client_id']
clients_in_val = X_val['client_id']

invoice_train_res = invoice_train[invoice_train['client_id'].isin(clients_in_resampled_train)]
invoice_val = invoice_train[invoice_train['client_id'].isin(clients_in_val)]

client_train_res = pd.merge(X_train_res, y_train_res, left_index=True, right_index=True)

In [11]:
print(X_train_res.shape, y_train_res.shape)
print(X_val.shape, y_val.shape)

(108394, 5) (108394,)
(27099, 5) (27099,)


## Feature Engineering Functions

In [12]:
#convert the column invoice_date to date time format on both the invoice train and invoice test
def invoice_convert_date(df):
    df_updated = df.copy()
    df_updated['invoice_date'] = pd.to_datetime(df_updated['invoice_date'])
    return df_updated

In [13]:
# convert tarif_type, counter_statue, counter_code, reading_remarque, counter_type to categorical data type
def invoice_to_category(df):
    df_updated = df.copy()
    df_updated['tarif_type'] = df_updated['tarif_type'].astype('category')
    df_updated['counter_statue'] = df_updated['counter_statue'].astype('category')
    df_updated['counter_code'] = df_updated['counter_code'].astype('category')
    df_updated['reading_remarque'] = df_updated['reading_remarque'].astype('category')
    df_updated['counter_type'] = df_updated['counter_type'].astype('category')
    return df_updated

In [14]:
# create variable that sums the consumption for all 4 levels
def invoice_create_consumption(df):
    df_updated = df.copy()
    df_updated['total_consumption'] = df_updated['new_index'] - df_updated['old_index']
    df_updated['consumption_per_month'] = df_updated['total_consumption'] / df_updated['months_number']
    return df_updated

In [15]:
# convert disrict, client_catg, region to categorical data type
def client_to_category(df):
    df_updated = df.copy()
    df_updated['disrict'] = df_updated['disrict'].astype('category')
    df_updated['client_catg'] = df_updated['client_catg'].astype('category')
    df_updated['region'] = df_updated['region'].astype('category')
    return df_updated

In [16]:
# convert creation_date to date time format on both the client train and client test
def client_convert_date(df):
    df_updated = df.copy()
    df_updated['creation_date'] = pd.to_datetime(df_updated['creation_date'], dayfirst=True)
    # create new variable with 7 bins for the creation date using cut
    df_updated['creation_date_bin'] = pd.cut(df_updated['creation_date'], bins=7, labels=False)
    # convert the creation_date_bin to categorical data type
    df_updated['creation_date_bin'] = df_updated['creation_date_bin'].astype('category')
    return df_updated

In [17]:
# DEFINE LEVEL USAGE
# -------------------------
def define_activity_level4_usage(df):
    df_updated = df.copy()
    df_updated['used_lvl4'] = (df_updated['consommation_level_4'] > 0).astype(int)
    # activity
    # any consumption at all (levels 1–4)
    df_updated['active'] = (
        (df_updated['consommation_level_1'] > 0) |
        (df_updated['consommation_level_2'] > 0) |
        (df_updated['consommation_level_3'] > 0) |
        (df_updated['consommation_level_4'] > 0)
    ).astype(int)
    return df_updated

In [18]:
# -------------------------
# FRACTION PER CLIENT
# -------------------------
def lvl4_fraction(df):
    df_updated = df.copy()
    client_lvl4_fraction = df_updated.groupby('client_id')['used_lvl4'].mean().reset_index()
    client_lvl4_fraction.rename(columns={
        'used_lvl4': 'frac_time_lvl4'
        }, inplace=True)
    df_updated['frac_time_lvl4'] = df_updated['client_id'].map(
        client_lvl4_fraction.set_index('client_id')['frac_time_lvl4']
    )
    return df_updated

In [19]:
def duplicate_invoices_same_day(df):
    df = df.copy()
    grp = df.groupby(['client_id', 'counter_type', 'invoice_date'])['client_id']
    # count rows per (client_id, counter_type, invoice_date)
    per_day_count = grp.transform('size')
    # per-row boolean: this row belongs to a duplicated same-day group
    duplicated_group = per_day_count.gt(1)
    # per-client flag: any duplicated same-day group for that client
    df['has_multi_invoice_same_day'] = duplicated_group.groupby(df['client_id']).transform('any').astype('int8')
    return df

In [20]:
# date gaps
def date_gaps(df):
    df_updated = df.copy()
    df_updated = df_updated.sort_values(['client_id', 'invoice_date'])
    df_updated['gap_days'] = df_updated.groupby(['client_id', 'counter_type'])['invoice_date'].diff().dt.days
    return df_updated

In [21]:
def invoice_scaled_consumption_per_month(df):
    df_updated = df.copy()
    df_updated['consumption_per_month_scaled'] = df_updated.groupby(['client_id', 'counter_type'])['consumption_per_month'].transform(lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() != x.min() else 0.5)
    return df_updated

In [22]:
def correct_newIndex(df):
    updated_df = df.copy()
    updated_df['index_diff'] = updated_df['new_index'] - updated_df['old_index']
    updated_df.loc[updated_df['index_diff'] < 0, 'new_index'] += 100000
    return updated_df

### Run on (undersampled) data

In [23]:
invoice_train_res = invoice_convert_date(invoice_train_res)
invoice_train_res = correct_newIndex(invoice_train_res)
invoice_train_res = invoice_to_category(invoice_train_res)
invoice_train_res = invoice_create_consumption(invoice_train_res)
invoice_train_res = define_activity_level4_usage(invoice_train_res)
invoice_train_res = lvl4_fraction(invoice_train_res)
invoice_train_res = duplicate_invoices_same_day(invoice_train_res)
invoice_train_res = date_gaps(invoice_train_res)
invoice_train_res = invoice_scaled_consumption_per_month(invoice_train_res)
# write the preprocessed invoice train_res to csv
invoice_train_res.to_csv('data/created_invoice_train_res.csv', index=False)

In [24]:
invoice_val = invoice_convert_date(invoice_val)
invoice_val = correct_newIndex(invoice_val)
invoice_val = invoice_to_category(invoice_val)
invoice_val = invoice_create_consumption(invoice_val)
invoice_val = define_activity_level4_usage(invoice_val)
invoice_val = lvl4_fraction(invoice_val)
invoice_val = duplicate_invoices_same_day(invoice_val)
invoice_val = date_gaps(invoice_val)
invoice_val = invoice_scaled_consumption_per_month(invoice_val)
# write the preprocessed invoice val to csv
invoice_val.to_csv('data/created_invoice_val.csv', index=False)

In [25]:
invoice_test = correct_newIndex(invoice_test)
invoice_test = invoice_convert_date(invoice_test)
invoice_test = invoice_to_category(invoice_test)
invoice_test = invoice_create_consumption(invoice_test)
invoice_test = define_activity_level4_usage(invoice_test)
invoice_test = lvl4_fraction(invoice_test)
invoice_test = duplicate_invoices_same_day(invoice_test)
invoice_test = date_gaps(invoice_test)
invoice_test = invoice_scaled_consumption_per_month(invoice_test)
# write the preprocessed invoice test to csv
invoice_test.to_csv('data/created_invoice_test.csv', index=False)

In [26]:
for df in [client_train_res, client_val, client_test]:
    df = client_convert_date(df)
    df = client_to_category(df)

# client_train_res.to_csv('data/created_client_train_res.csv', index=False)
# client_val.to_csv('data/created_client_val.csv', index=False)
# client_test.to_csv('data/created_client_test.csv', index=False)

In [27]:
# read created train_res data
# invoice_train_res = pd.read_csv('data/created_invoice_train_res.csv')
# read created data
# invoice_val = pd.read_csv('data/created_invoice_val.csv')
# read created data
# invoice_test = pd.read_csv('data/created_invoice_test.csv')

# read created client data
# client_train_res = pd.read_csv('data/created_client_train_res.csv')
# client_val = pd.read_csv('data/created_client_val.csv')
# client_test = pd.read_csv('data/created_client_test.csv')

In [28]:
invoice_train_res.tail()

,client_id,invoice_date,tarif_type,counter_number,counter_statue,counter_code,reading_remarque,counter_coefficient,consommation_level_1,consommation_level_2,...,counter_type,index_diff,total_consumption,consumption_per_month,used_lvl4,active,frac_time_lvl4,has_multi_invoice_same_day,gap_days,consumption_per_month_scaled
4476744,train_Client_99998,2005-08-19,10,1253571,0,202,9,1,400,135,...,ELEC,535,535,66.875,0,1,0.0,0,NaN,1.000000
4476745,train_Client_99998,2005-12-19,10,1253571,0,202,6,1,200,6,...,ELEC,206,206,51.500,0,1,0.0,0,122.0,0.000000
4476748,train_Client_99999,1996-01-25,11,560948,0,203,6,1,516,0,...,ELEC,516,516,129.000,0,1,0.0,0,NaN,0.747093
4476747,train_Client_99999,1996-05-28,11,560948,0,203,6,1,603,0,...,ELEC,603,603,150.750,0,1,0.0,0,124.0,1.000000
4476746,train_Client_99999,1996-09-25,11,560948,0,203,6,1,259,0,...,ELEC,259,259,64.750,0,1,0.0,0,120.0,0.000000


## Aggregate functions on invoice to client

In [29]:
def aggregate_by_client_id(invoice_data):
    aggs = {}
    aggs['gap_days'] = [np.nanmean]
    aggs['gap_days'] = [np.nanstd]
    aggs['consumption_per_month'] = [np.nanmean]
    aggs['consumption_per_month'] = [np.nanstd]
    aggs['frac_time_lvl4'] = [np.nanmax]
    aggs['consumption_per_month_scaled'] = [np.nanstd]
    aggs['active'] = ['sum']
    aggs['active'] = [np.nanmean]
    aggs['counter_statue'] = [np.nanstd]
    aggs['reading_remarque'] = [np.nanstd]
    aggs['tarif_type'] = [scipy.stats.mode]
    aggs['index_diff'] = [np.nanstd]
    aggs['has_multi_invoice_same_day'] = [np.nanmax]
    aggs['used_lvl4'] = ['sum']


    agg_trans = invoice_data.groupby(['client_id']).agg(aggs)
    agg_trans.columns = ['_'.join(col).strip() for col in agg_trans.columns.values]
    agg_trans.reset_index(inplace=True)

    df = (invoice_data.groupby('client_id')
            .size()
            .reset_index(name='{}transactions_count'.format('1')))
    return pd.merge(df, agg_trans, on='client_id', how='left')

In [30]:
#group invoice data by client_id
agg_train_res = aggregate_by_client_id(invoice_train_res)
agg_val = aggregate_by_client_id(invoice_val)
agg_test = aggregate_by_client_id(invoice_test)

In [31]:
agg_train_res.head()

,client_id,1transactions_count,gap_days_nanstd,consumption_per_month_nanstd,frac_time_lvl4_nanmax,consumption_per_month_scaled_nanstd,active_nanmean,counter_statue_nanstd,reading_remarque_nanstd,tarif_type_mode,index_diff_nanstd,has_multi_invoice_same_day_nanmax,used_lvl4_sum
0,train_Client_0,35,87.601070,84.545640,0.000000,0.258155,1.000,0.000000,1.248192,"(11, 35)",341.553930,0,0
1,train_Client_1,37,92.492312,24.105605,0.000000,0.214272,1.000,0.000000,1.377097,"(11, 37)",197.935960,0,0
2,train_Client_100,20,95.569974,0.901753,0.000000,0.240467,0.200,0.000000,0.670820,"(11, 20)",3.607011,0,0
3,train_Client_1000,14,80.057752,150.969514,0.142857,0.282979,1.000,0.000000,0.363137,"(11, 14)",633.485669,0,2
4,train_Client_100000,40,46.310659,89.272755,0.000000,0.264030,0.925,0.405096,1.165476,"(11, 20)",338.205437,0,0


In [32]:
# replace tarif_type_mode by first entry of that column
agg_train_res['tarif_type_mode'] = agg_train_res['tarif_type_mode'].apply(lambda x: x[0] if isinstance(x, tuple) else x)
# change to category type
agg_train_res['tarif_type_mode'] = agg_train_res['tarif_type_mode'].astype('category')

# same for val and test
agg_val['tarif_type_mode'] = agg_val['tarif_type_mode'].apply(lambda x: x[0] if isinstance(x, tuple) else x)
agg_val['tarif_type_mode'] = agg_val['tarif_type_mode'].astype('category')
agg_test['tarif_type_mode'] = agg_test['tarif_type_mode'].apply(lambda x: x[0] if isinstance(x, tuple) else x)
agg_test['tarif_type_mode'] = agg_test['tarif_type_mode'].astype('category')

In [33]:
#merge aggregate data with client dataset
train_res = pd.merge(client_train_res, agg_train_res, on='client_id', how='left')
display(train_res)

train_res.to_csv('data/created_train_res.csv', index=False)

#same for val 
val = pd.merge(client_val, agg_val, on='client_id', how='left')
display(val)

val.to_csv('data/created_val.csv', index=False)

# merge client test with agg test
test = pd.merge(client_test, agg_test, on='client_id', how='left')
display(test)

test.to_csv('data/created_test.csv', index=False)

,disrict,client_id,client_catg,region,creation_date,target,1transactions_count,gap_days_nanstd,consumption_per_month_nanstd,frac_time_lvl4_nanmax,consumption_per_month_scaled_nanstd,active_nanmean,counter_statue_nanstd,reading_remarque_nanstd,tarif_type_mode,index_diff_nanstd,has_multi_invoice_same_day_nanmax,used_lvl4_sum
0,69,train_Client_46032,11,105,25/12/2015,0.0,22.0,23.611772,368.722424,0.272727,0.373383,1.000000,0.000000,0.492366,11,1514.320916,0.0,6.0
1,69,train_Client_8884,11,107,13/11/2010,1.0,14.0,90.176930,219.425698,0.000000,0.259113,0.500000,0.000000,1.414214,11,967.714530,0.0,0.0
2,60,train_Client_80860,11,101,28/10/2008,0.0,31.0,124.129354,17.990506,0.000000,0.240676,1.000000,0.179605,1.408881,11,109.810061,0.0,0.0
3,60,train_Client_21716,11,101,29/10/1984,0.0,48.0,52.741499,57.583573,0.000000,0.254952,0.979167,0.000000,1.368173,11,339.124645,0.0,0.0
4,62,train_Client_95171,11,301,24/03/1989,0.0,65.0,80.480377,43.084777,0.000000,0.268246,0.984615,0.291712,1.380322,11,155.234802,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108389,62,train_Client_107750,11,301,20/11/1997,0.0,42.0,88.326528,93.750470,0.000000,0.260780,1.000000,0.000000,1.380341,11,375.001879,0.0,0.0
108390,69,train_Client_59063,11,104,10/06/1998,0.0,67.0,62.270760,59.476032,0.000000,0.282977,0.955224,0.621127,1.280265,11,261.842966,0.0,0.0
108391,60,train_Client_68496,11,101,12/07/2008,1.0,34.0,68.256576,84.191740,0.000000,0.282049,0.852941,0.171499,1.299938,11,338.327368,0.0,0.0
108392,62,train_Client_56302,11,309,09/12/2013,0.0,27.0,72.449983,50.954021,0.000000,0.329893,1.000000,0.000000,0.974022,40,203.458999,0.0,0.0


,disrict,client_id,client_catg,region,creation_date,target,1transactions_count,gap_days_nanstd,consumption_per_month_nanstd,frac_time_lvl4_nanmax,consumption_per_month_scaled_nanstd,active_nanmean,counter_statue_nanstd,reading_remarque_nanstd,tarif_type_mode,index_diff_nanstd,has_multi_invoice_same_day_nanmax,used_lvl4_sum
0,69,train_Client_15159,11,104,24/12/2008,0.0,26.0,71.863714,69.225383,0.000000,0.284293,0.923077,0.196116,1.336471,11,288.338380,0.0,0.0
1,63,train_Client_80740,11,311,13/09/2014,0.0,30.0,53.249537,49.349953,0.000000,0.291712,1.000000,0.000000,0.547723,11,169.041004,0.0,0.0
2,63,train_Client_59100,11,311,07/11/1989,1.0,55.0,119.886288,161.799264,0.054545,0.345812,0.763636,0.134840,1.361570,10,1003.806981,0.0,3.0
3,69,train_Client_125675,11,105,10/11/1993,0.0,58.0,135.559102,149.164036,0.000000,0.275661,0.982759,0.000000,1.308911,11,715.475605,0.0,0.0
4,63,train_Client_4885,11,313,29/12/2012,0.0,32.0,86.278352,36.327523,0.000000,0.258478,0.500000,0.296145,1.524002,11,144.914644,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27094,62,train_Client_106726,11,304,16/11/1998,0.0,43.0,37.224965,3.021254,0.000000,0.151063,0.976744,1.071115,1.096658,11,7.450910,1.0,0.0
27095,60,train_Client_83056,11,101,28/02/1992,0.0,26.0,173.623184,43.246750,0.000000,0.279011,0.923077,0.000000,1.250846,11,409.988099,0.0,0.0
27096,69,train_Client_92212,11,104,22/11/2008,0.0,31.0,98.590439,367.482570,0.516129,0.187563,1.000000,0.179605,1.469401,11,1465.477179,0.0,16.0
27097,63,train_Client_76137,11,312,28/03/2002,0.0,43.0,47.695413,32.841130,0.000000,0.247713,0.976744,0.762493,1.365206,11,131.364519,0.0,0.0


,disrict,client_id,client_catg,region,creation_date,1transactions_count,gap_days_nanstd,consumption_per_month_nanstd,frac_time_lvl4_nanmax,consumption_per_month_scaled_nanstd,active_nanmean,counter_statue_nanstd,reading_remarque_nanstd,tarif_type_mode,index_diff_nanstd,has_multi_invoice_same_day_nanmax,used_lvl4_sum
0,62,test_Client_0,11,307,28/05/2002,37.0,44.900834,40.424028,0.000000,0.175757,0.972973,0.000000,1.221061,11,235.684859,0.0,0.0
1,69,test_Client_1,11,103,06/08/2009,22.0,117.157647,722.359649,0.045455,0.194222,1.000000,0.213201,1.216766,11,3200.849986,0.0,1.0
2,62,test_Client_10,11,310,07/04/2004,74.0,13.134901,105.694968,0.013514,0.241883,0.986486,0.000000,1.482216,11,422.779872,0.0,1.0
3,60,test_Client_100,11,101,08/10/1992,40.0,96.299793,66.060236,0.000000,0.293850,0.950000,0.000000,1.034966,11,247.253171,0.0,0.0
4,62,test_Client_1000,11,301,21/07/1977,53.0,113.376261,158.358003,0.000000,0.280807,0.924528,0.686803,1.319443,11,864.294814,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58064,63,test_Client_9995,11,399,17/03/2010,4.0,0.000000,133.188991,0.000000,0.408248,0.500000,0.000000,0.000000,11,532.755964,0.0,0.0
58065,63,test_Client_9996,11,311,28/05/2011,46.0,58.724011,23.653269,0.000000,0.220683,0.978261,0.759163,1.100066,11,107.556838,0.0,0.0
58066,60,test_Client_9997,11,101,04/03/1978,59.0,120.360325,31.634983,0.000000,0.220892,1.000000,0.000000,1.065094,10,116.987086,0.0,0.0
58067,60,test_Client_9998,11,101,23/02/2018,1.0,NaN,NaN,0.000000,NaN,1.000000,NaN,NaN,11,NaN,0.0,0.0


In [34]:
train_res = pd.merge(client_train_res, agg_train_res, on='client_id', how='left')
display(train_res)

train_res.to_csv('data/created_train_res.csv', index=False)

#same for val 
val = pd.merge(client_val, agg_val, on='client_id', how='left')
display(val)

val.to_csv('data/created_val.csv', index=False)

# merge client test with agg test
test = pd.merge(client_test, agg_test, on='client_id', how='left')
display(test)

test.to_csv('data/created_test.csv', index=False)

,disrict,client_id,client_catg,region,creation_date,target,1transactions_count,gap_days_nanstd,consumption_per_month_nanstd,frac_time_lvl4_nanmax,consumption_per_month_scaled_nanstd,active_nanmean,counter_statue_nanstd,reading_remarque_nanstd,tarif_type_mode,index_diff_nanstd,has_multi_invoice_same_day_nanmax,used_lvl4_sum
0,69,train_Client_46032,11,105,25/12/2015,0.0,22.0,23.611772,368.722424,0.272727,0.373383,1.000000,0.000000,0.492366,11,1514.320916,0.0,6.0
1,69,train_Client_8884,11,107,13/11/2010,1.0,14.0,90.176930,219.425698,0.000000,0.259113,0.500000,0.000000,1.414214,11,967.714530,0.0,0.0
2,60,train_Client_80860,11,101,28/10/2008,0.0,31.0,124.129354,17.990506,0.000000,0.240676,1.000000,0.179605,1.408881,11,109.810061,0.0,0.0
3,60,train_Client_21716,11,101,29/10/1984,0.0,48.0,52.741499,57.583573,0.000000,0.254952,0.979167,0.000000,1.368173,11,339.124645,0.0,0.0
4,62,train_Client_95171,11,301,24/03/1989,0.0,65.0,80.480377,43.084777,0.000000,0.268246,0.984615,0.291712,1.380322,11,155.234802,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108389,62,train_Client_107750,11,301,20/11/1997,0.0,42.0,88.326528,93.750470,0.000000,0.260780,1.000000,0.000000,1.380341,11,375.001879,0.0,0.0
108390,69,train_Client_59063,11,104,10/06/1998,0.0,67.0,62.270760,59.476032,0.000000,0.282977,0.955224,0.621127,1.280265,11,261.842966,0.0,0.0
108391,60,train_Client_68496,11,101,12/07/2008,1.0,34.0,68.256576,84.191740,0.000000,0.282049,0.852941,0.171499,1.299938,11,338.327368,0.0,0.0
108392,62,train_Client_56302,11,309,09/12/2013,0.0,27.0,72.449983,50.954021,0.000000,0.329893,1.000000,0.000000,0.974022,40,203.458999,0.0,0.0


,disrict,client_id,client_catg,region,creation_date,target,1transactions_count,gap_days_nanstd,consumption_per_month_nanstd,frac_time_lvl4_nanmax,consumption_per_month_scaled_nanstd,active_nanmean,counter_statue_nanstd,reading_remarque_nanstd,tarif_type_mode,index_diff_nanstd,has_multi_invoice_same_day_nanmax,used_lvl4_sum
0,69,train_Client_15159,11,104,24/12/2008,0.0,26.0,71.863714,69.225383,0.000000,0.284293,0.923077,0.196116,1.336471,11,288.338380,0.0,0.0
1,63,train_Client_80740,11,311,13/09/2014,0.0,30.0,53.249537,49.349953,0.000000,0.291712,1.000000,0.000000,0.547723,11,169.041004,0.0,0.0
2,63,train_Client_59100,11,311,07/11/1989,1.0,55.0,119.886288,161.799264,0.054545,0.345812,0.763636,0.134840,1.361570,10,1003.806981,0.0,3.0
3,69,train_Client_125675,11,105,10/11/1993,0.0,58.0,135.559102,149.164036,0.000000,0.275661,0.982759,0.000000,1.308911,11,715.475605,0.0,0.0
4,63,train_Client_4885,11,313,29/12/2012,0.0,32.0,86.278352,36.327523,0.000000,0.258478,0.500000,0.296145,1.524002,11,144.914644,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27094,62,train_Client_106726,11,304,16/11/1998,0.0,43.0,37.224965,3.021254,0.000000,0.151063,0.976744,1.071115,1.096658,11,7.450910,1.0,0.0
27095,60,train_Client_83056,11,101,28/02/1992,0.0,26.0,173.623184,43.246750,0.000000,0.279011,0.923077,0.000000,1.250846,11,409.988099,0.0,0.0
27096,69,train_Client_92212,11,104,22/11/2008,0.0,31.0,98.590439,367.482570,0.516129,0.187563,1.000000,0.179605,1.469401,11,1465.477179,0.0,16.0
27097,63,train_Client_76137,11,312,28/03/2002,0.0,43.0,47.695413,32.841130,0.000000,0.247713,0.976744,0.762493,1.365206,11,131.364519,0.0,0.0


,disrict,client_id,client_catg,region,creation_date,1transactions_count,gap_days_nanstd,consumption_per_month_nanstd,frac_time_lvl4_nanmax,consumption_per_month_scaled_nanstd,active_nanmean,counter_statue_nanstd,reading_remarque_nanstd,tarif_type_mode,index_diff_nanstd,has_multi_invoice_same_day_nanmax,used_lvl4_sum
0,62,test_Client_0,11,307,28/05/2002,37.0,44.900834,40.424028,0.000000,0.175757,0.972973,0.000000,1.221061,11,235.684859,0.0,0.0
1,69,test_Client_1,11,103,06/08/2009,22.0,117.157647,722.359649,0.045455,0.194222,1.000000,0.213201,1.216766,11,3200.849986,0.0,1.0
2,62,test_Client_10,11,310,07/04/2004,74.0,13.134901,105.694968,0.013514,0.241883,0.986486,0.000000,1.482216,11,422.779872,0.0,1.0
3,60,test_Client_100,11,101,08/10/1992,40.0,96.299793,66.060236,0.000000,0.293850,0.950000,0.000000,1.034966,11,247.253171,0.0,0.0
4,62,test_Client_1000,11,301,21/07/1977,53.0,113.376261,158.358003,0.000000,0.280807,0.924528,0.686803,1.319443,11,864.294814,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58064,63,test_Client_9995,11,399,17/03/2010,4.0,0.000000,133.188991,0.000000,0.408248,0.500000,0.000000,0.000000,11,532.755964,0.0,0.0
58065,63,test_Client_9996,11,311,28/05/2011,46.0,58.724011,23.653269,0.000000,0.220683,0.978261,0.759163,1.100066,11,107.556838,0.0,0.0
58066,60,test_Client_9997,11,101,04/03/1978,59.0,120.360325,31.634983,0.000000,0.220892,1.000000,0.000000,1.065094,10,116.987086,0.0,0.0
58067,60,test_Client_9998,11,101,23/02/2018,1.0,NaN,NaN,0.000000,NaN,1.000000,NaN,NaN,11,NaN,0.0,0.0


## Tips 
- Thorough EDA and incorporating domain knowledge
- Re-grouping categorical features
- More feature engineering(try utilizing some date-time features)
- Target balancing - oversampling, undersampling, SMOTE, scale_pos_weight
- Model ensembling
- Train-test split or cross-validation


# ******************* GOOD LUCK!!! ***************************